# Feature Engineering

Arya Bima Nugraha (238)

M. Haekal Fallah Ar-Rahman (061)

> *"Try applying all the feature engineering on the text you have preprocessed from the previous exercise and analyze the given results."*

Input: file `exercise4_text_preprocessed.csv` (hasil Exercise 4, kolom `prepro`). Upload file tersebut ke Colab (atau taruh satu folder dengan notebook ini) sebelum menjalankan.

Metode yang diterapkan:
- **Chapter 5**: One Hot Encoding, Bag of Words (count), TF-IDF
- **Chapter 6**: Word2Vec (CBOW) dan FastText (Skip-Gram) + visualisasi UMAP

## 1. Install & import library

In [38]:
!pip install gensim umap-learn -q

In [39]:
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import nltk
import plotly.express as px
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 2. Load data hasil pre-processing (Exercise 4)

In [40]:
df = pd.read_excel('Exercise4_Hasil_Preprocessing.xlsx')
df = df.dropna(subset=['prepro']).reset_index(drop=True)
df_pre = df['prepro']
df_pre.info()
df_pre

<class 'pandas.core.series.Series'>
RangeIndex: 8 entries, 0 to 7
Series name: prepro
Non-Null Count  Dtype 
--------------  ----- 
8 non-null      object
dtypes: object(1)
memory usage: 196.0+ bytes


,prepro
0,first love letter 1991 movi detail direct reno...
1,journalist anonym confess secret feel love let...
2,data de lançamento 25 de março de 1995 japão
3,anoth rare time travel movi sort avail anywher...
4,direct shunji iwai miho nakayama etsushi toyokawa
5,first love letter 1991 superhit hindi romant m...
6,director shunji iwai writer shunji iwai star i...
7,theater june 5 hiroko still depth grief year f...


## 3. Chapter 5 — Feature Engineering Part 1

### 3.1 One Hot Encoding
Setiap kata bernilai 1 jika muncul di dokumen (tanpa peduli jumlah kemunculan), selain itu 0.

In [41]:
vec_ohe = CountVectorizer(binary=True)
X_ohe = vec_ohe.fit_transform(df_pre)
df_ohe = pd.DataFrame(X_ohe.toarray(), columns=vec_ohe.get_feature_names_out())
print('Shape:', df_ohe.shape)
df_ohe

Shape: (8, 65)


,1991,1995,25,accid,anonym,anoth,anywher,avail,best,climb,...,star,still,stori,superhit,theater,time,toyokawa,travel,writer,year
0,1,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,1,1,0,0,...,0,0,0,0,0,1,0,1,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
5,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
7,0,0,0,1,0,0,0,0,0,1,...,0,1,0,0,1,0,0,0,0,1


### 3.2 Bag of Words (Count)
Sama seperti One Hot Encoding, tetapi nilainya adalah jumlah kemunculan kata di dokumen.

In [42]:
vec_bow = CountVectorizer()
X_bow = vec_bow.fit_transform(df_pre)
df_bow = pd.DataFrame(X_bow.toarray(), columns=vec_bow.get_feature_names_out())
print('Shape:', df_bow.shape)
df_bow

Shape: (8, 65)


,1991,1995,25,accid,anonym,anoth,anywher,avail,best,climb,...,star,still,stori,superhit,theater,time,toyokawa,travel,writer,year
0,1,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,1,1,0,0,...,0,0,0,0,0,1,0,1,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
5,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
7,0,0,0,1,0,0,0,0,0,1,...,0,1,0,0,1,0,0,0,0,1


In [43]:
diff = (df_bow != df_ohe).any(axis=0)
print('Kata dengan nilai berbeda antara BoW dan OHE:', list(df_bow.columns[diff]))

df_bow.sum().sort_values(ascending=False).head(10)

Kata dengan nilai berbeda antara BoW dan OHE: ['de', 'iwai', 'shunji']


,0
de,3
movi,3
letter,3
love,3
shunji,3
iwai,3
1991,2
first,2
direct,2
romant,2


### 3.3 TF-IDF
Kata yang sering muncul di satu dokumen tetapi jarang di dokumen lain diberi bobot lebih tinggi.

In [44]:
vec_tfidf = TfidfVectorizer()
X_tfidf = vec_tfidf.fit_transform(df_pre)
df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=vec_tfidf.get_feature_names_out())
print('Shape:', df_tfidf.shape)
df_tfidf

Shape: (8, 65)


,1991,1995,25,accid,anonym,anoth,anywher,avail,best,climb,...,star,still,stori,superhit,theater,time,toyokawa,travel,writer,year
0,0.251766,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.300409,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.332484,0.000000,0.000000,0.000000,0.332484,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
2,0.000000,0.258199,0.258199,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.324051,0.324051,0.324051,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.324051,0.000000,0.324051,0.00000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.404652,0.000000,0.00000,0.000000
5,0.289535,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.345475,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.29337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.29337,0.000000
7,0.000000,0.000000,0.000000,0.301511,0.000000,0.000000,0.000000,0.000000,0.000000,0.301511,...,0.00000,0.301511,0.000000,0.000000,0.301511,0.000000,0.000000,0.000000,0.00000,0.301511


In [45]:
for i, row in df_tfidf.iterrows():
    top = row.sort_values(ascending=False).head(3)
    print(i, df.loc[i, 'video_id'], '->', ', '.join(f'{w} ({s:.2f})' for w, s in top.items()))

0 SrvLb03AXok -> detail (0.30), stori (0.30), heartfelt (0.30)
1 bDe5W2I8Sgc -> anonym (0.33), friend (0.33), best (0.33)
2 ffuFHYh6TWs -> de (0.77), lançamento (0.26), 1995 (0.26)
3 WsKyc8gJW1k -> anywher (0.32), travel (0.32), avail (0.32)
4 P6JAtpSC0EA -> etsushi (0.40), miho (0.40), toyokawa (0.40)
5 sV9JaCWTDgE -> full (0.35), emot (0.35), hindi (0.35)
6 V_tCZB4YRbE -> shunji (0.49), iwai (0.49), icon (0.29)
7 tvneMVSrvbU -> accid (0.30), still (0.30), die (0.30)


In [46]:
cmp = pd.DataFrame({
    'bow_total': df_bow.sum(),
    'df (jumlah dokumen)': (df_bow > 0).sum(),
    'idf': pd.Series(vec_tfidf.idf_, index=vec_tfidf.get_feature_names_out()),
    'tfidf_mean': df_tfidf.mean()
})
cmp.sort_values('bow_total', ascending=False).head(10)

,bow_total,df (jumlah dokumen),idf,tfidf_mean
de,3,1,2.504077,0.096825
movi,3,3,1.810930,0.087681
letter,3,3,1.810930,0.088444
love,3,3,1.810930,0.088444
shunji,3,2,2.098612,0.103858
iwai,3,2,2.098612,0.103858
1991,2,2,2.098612,0.067663
first,2,2,2.098612,0.067663
direct,2,2,2.098612,0.073862
romant,2,2,2.098612,0.067663


## 4. Chapter 6 — Feature Engineering Part 2 (Word Embedding)

### 4.1 Tokenize

In [47]:
sentences = [word_tokenize(text.lower()) for text in df_pre]
sentences

[['first',
  'love',
  'letter',
  '1991',
  'movi',
  'detail',
  'direct',
  'renown',
  'filmmak',
  'romant',
  'drama',
  'follow',
  'heartfelt',
  'stori'],
 ['journalist',
  'anonym',
  'confess',
  'secret',
  'feel',
  'love',
  'letter',
  'sent',
  'best',
  'friend'],
 ['data', 'de', 'lançamento', '25', 'de', 'março', 'de', '1995', 'japão'],
 ['anoth',
  'rare',
  'time',
  'travel',
  'movi',
  'sort',
  'avail',
  'anywher',
  'els',
  'onlin'],
 ['direct', 'shunji', 'iwai', 'miho', 'nakayama', 'etsushi', 'toyokawa'],
 ['first',
  'love',
  'letter',
  '1991',
  'superhit',
  'hindi',
  'romant',
  'movi',
  'full',
  'drama',
  'emot'],
 ['director',
  'shunji',
  'iwai',
  'writer',
  'shunji',
  'iwai',
  'star',
  'icon',
  'mountain',
  'scene'],
 ['theater',
  'june',
  '5',
  'hiroko',
  'still',
  'depth',
  'grief',
  'year',
  'fiancé',
  'die',
  'climb',
  'accid']]

### 4.2 Word2Vec (CBOW)

Parameter:
* `min_count=1`: semua kata disertakan (korpus sangat kecil).
* `vector_size=100`: dimensi vektor kata.
* `window=5`: konteks 5 kata kiri-kanan.
* `sg=0`: CBOW.
* `epochs=100`, `seed=42`, `workers=1`: korpus hanya beberapa dokumen, sehingga dilatih lebih lama dan dibuat reproducible (default `epochs=5` terlalu sedikit untuk data sekecil ini).

In [48]:
import gensim.models

model = gensim.models.Word2Vec(sentences, min_count=1, vector_size=100, window=5, sg=0,
                               epochs=100, seed=42, workers=1)

In [49]:
model.save('youtube_text.w2v')

model = gensim.models.Word2Vec.load('youtube_text.w2v')

In [50]:
w2v = model.wv
print('Jumlah kata (vocab):', len(w2v.index_to_key))
print('10 kata pertama   :', w2v.index_to_key[:10])
print('Shape vektor      :', w2v.vectors.shape, '| vector_size =', w2v.vector_size)

Jumlah kata (vocab): 66
10 kata pertama   : ['iwai', 'shunji', 'de', 'movi', 'letter', 'love', 'drama', 'romant', 'direct', '1991']
Shape vektor      : (66, 100) | vector_size = 100


In [51]:
w2v.similar_by_key('love')

[('mountain', 0.2993665039539337),
 ('anonym', 0.21908390522003174),
 ('secret', 0.1891263723373413),
 ('lançamento', 0.17903947830200195),
 ('follow', 0.17901501059532166),
 ('stori', 0.1784755140542984),
 ('março', 0.15837770700454712),
 ('die', 0.1509595811367035),
 ('1991', 0.13918963074684143),
 ('hiroko', 0.13350878655910492)]

In [52]:
def get_document_vector(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_vectors:
        return np.zeros(model.vector_size)
    return np.mean(valid_vectors, axis=0)

X_w2v = np.array([get_document_vector(doc, model) for doc in sentences])
pd.DataFrame(X_w2v)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.000687,0.005416,-0.003319,0.003123,-0.002375,0.002916,-0.001030,0.001852,-0.001384,-0.002065,...,0.000567,0.002751,-0.001226,0.001413,-0.000595,-0.000647,0.002037,0.002013,-0.003106,-0.003681
1,0.001308,0.003206,-0.003681,0.004372,-0.004464,0.000261,-0.001438,0.003591,-0.000455,-0.002095,...,0.000086,0.002012,0.000470,0.001482,0.001582,-0.003115,0.000249,0.005141,0.003934,-0.000403
2,0.000755,0.004144,-0.002209,0.005233,-0.002353,0.001248,0.000506,0.009518,-0.000755,0.001691,...,0.002861,-0.000240,0.005121,-0.002859,0.001200,-0.002698,0.005423,0.000937,-0.000398,-0.004930
3,-0.000939,0.005130,-0.004351,-0.001261,-0.002560,0.004684,-0.003361,0.005814,-0.004052,0.000713,...,-0.001587,-0.000916,-0.002675,-0.004555,0.004391,0.000506,-0.000493,-0.002969,-0.002114,-0.003090
4,-0.001451,0.001953,-0.001834,0.000285,-0.001428,0.002613,-0.001638,0.002588,-0.004204,0.002017,...,-0.004544,0.002470,0.002894,-0.000698,0.003670,-0.001362,-0.002903,0.000577,-0.005782,0.001301
5,0.001384,0.004796,-0.001348,0.003321,-0.002003,0.002281,0.000019,0.002128,-0.000079,-0.003240,...,-0.000178,0.002149,0.000945,0.000609,0.000503,-0.000362,-0.000662,0.002429,-0.001077,-0.002577
6,0.001197,-0.000169,0.003262,-0.002031,-0.001852,0.003807,-0.002072,0.002123,-0.002038,-0.001980,...,-0.000778,0.000409,0.004871,-0.002494,0.003144,-0.003561,0.001014,0.001809,-0.000924,-0.001208
7,0.002276,-0.000631,-0.000742,-0.002299,-0.001916,0.000291,-0.002972,0.003967,-0.004590,-0.000135,...,-0.001221,0.000486,-0.000194,-0.003306,0.000936,-0.000061,0.002461,0.006373,-0.000606,-0.001827


### 4.3 Visualisasi Word2Vec (UMAP)

In [53]:
from umap import UMAP

n_neighbors = min(15, len(w2v.index_to_key) - 1)
X = UMAP(n_neighbors=n_neighbors, random_state=42).fit_transform(w2v.vectors)

df_umap = pd.DataFrame(X, columns=['umap1', 'umap2'])
df_umap['word'] = w2v.index_to_key

fig = px.scatter(df_umap, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=800, title_text='Word2Vec Visualization')
fig.show()

### 4.4 FastText (Skip-Gram)

Parameter:
* `vector_size=100`, `window=5`: sama seperti Word2Vec.
* `min_n=3`, `max_n=6`: FastText memecah kata menjadi character n-gram 3–6 huruf.
* `sg=1`: Skip-Gram.
* `epochs=100`, `seed=42`, `workers=1`: alasan sama seperti Word2Vec.

In [54]:
model_fasttext = gensim.models.FastText(sentences, min_count=1, vector_size=100, window=5,
                                        min_n=3, max_n=6, sg=1, epochs=100,
                                        seed=42, workers=1)

model_fasttext.save('youtube_text.fasttext')

model_fasttext = gensim.models.FastText.load('youtube_text.fasttext')

In [55]:
fastext = model_fasttext.wv
print('Jumlah kata (vocab):', len(fastext.index_to_key))
print('Shape vektor      :', fastext.vectors.shape)
fastext.similar_by_key('love')

Jumlah kata (vocab): 66
Shape vektor      : (66, 100)


[('heartfelt', 0.8461031317710876),
 ('june', 0.8370893001556396),
 ('theater', 0.8345988392829895),
 ('renown', 0.8310595154762268),
 ('depth', 0.8309406638145447),
 ('romant', 0.8256153464317322),
 ('year', 0.8246455192565918),
 ('drama', 0.8198789358139038),
 ('detail', 0.818126916885376),
 ('hiroko', 0.8133376836776733)]

In [56]:
print('Word2Vec  punya "loving" di vocab?', 'loving' in w2v.key_to_index)
print('FastText  punya "loving" di vocab?', 'loving' in fastext.key_to_index)
print('Vektor FastText untuk "loving" tetap bisa dibuat, shape:', fastext['loving'].shape)
fastext.most_similar('loving', topn=5)

Word2Vec  punya "loving" di vocab? False
FastText  punya "loving" di vocab? False
Vektor FastText untuk "loving" tetap bisa dibuat, shape: (100,)


[('stori', 0.6345345973968506),
 ('anonym', 0.6133162975311279),
 ('depth', 0.609268844127655),
 ('nakayama', 0.6031088829040527),
 ('writer', 0.5931738615036011)]

In [57]:
# 5. FE result
X_ft = np.array([get_document_vector(doc, model_fasttext) for doc in sentences])
pd.DataFrame(X_ft)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.010189,0.001911,0.003446,0.007898,0.006623,-0.001593,-0.000476,0.002095,0.005247,0.005099,...,0.003503,-0.004520,0.000120,0.002628,-0.004536,-0.002206,0.002883,-0.000614,0.002146,-0.000781
1,0.008334,0.002131,0.002284,0.006177,0.005813,-0.000805,-0.000681,0.002761,0.004285,0.004479,...,0.002675,-0.003205,0.000603,0.001278,-0.004446,-0.001098,0.001597,-0.000796,0.001988,0.000428
2,0.011408,0.003858,0.005989,0.007171,0.007228,-0.002108,-0.000856,0.002788,0.005559,0.007534,...,0.006872,-0.005458,0.003098,0.001814,-0.003847,-0.003510,0.003116,-0.001719,0.001751,0.000530
3,0.009649,0.001712,0.001780,0.007607,0.006653,-0.001000,-0.000473,0.002416,0.004443,0.004751,...,0.002408,-0.004767,0.001140,0.002182,-0.005105,-0.001070,0.002554,-0.001781,0.002004,-0.000609
4,0.006154,0.000966,0.001196,0.004325,0.004438,-0.000435,-0.001771,0.001656,0.003693,0.004135,...,0.002378,-0.003037,0.000670,0.000993,-0.002832,-0.001259,0.001923,-0.000996,0.001074,0.000289
5,0.008843,0.002452,0.003167,0.006920,0.005423,-0.001421,-0.000342,0.002290,0.004364,0.004563,...,0.002815,-0.004309,0.000409,0.002378,-0.003572,-0.001776,0.002224,-0.000496,0.002065,-0.000867
6,0.008708,0.001208,0.001311,0.005964,0.004930,-0.000020,-0.000449,0.002563,0.005516,0.004761,...,0.003150,-0.003144,0.000626,0.001006,-0.003242,-0.000927,0.002284,-0.001583,0.001248,-0.000461
7,0.013793,0.001731,0.003062,0.009922,0.008352,-0.001890,-0.000195,0.002563,0.007943,0.008263,...,0.006469,-0.005088,0.000591,0.002164,-0.006830,-0.001529,0.004525,-0.000312,0.004389,0.000208


In [58]:
n_neighbors = min(15, len(fastext.index_to_key) - 1)
X = UMAP(n_neighbors=n_neighbors, random_state=42).fit_transform(fastext.vectors)

df_umap2 = pd.DataFrame(X, columns=['umap1', 'umap2'])
df_umap2['word'] = fastext.index_to_key

fig = px.scatter(df_umap2, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=800, title_text='FastText Visualization')
fig.show()

## 5. Ringkasan

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

def mean_pairwise_cosine(X):
    sim = cosine_similarity(X)
    n = sim.shape[0]
    return (sim.sum() - n) / (n * (n - 1))

for name, M in [('One Hot Encoding', df_ohe.values), ('Bag of Words', df_bow.values), ('TF-IDF', df_tfidf.values),
                ('Word2Vec', X_w2v), ('FastText', X_ft)]:
    print(f'{name:18s} rata-rata cosine similarity antar dokumen = {mean_pairwise_cosine(M):.3f}')

One Hot Encoding   rata-rata cosine similarity antar dokumen = 0.053
Bag of Words       rata-rata cosine similarity antar dokumen = 0.057
TF-IDF             rata-rata cosine similarity antar dokumen = 0.043
Word2Vec           rata-rata cosine similarity antar dokumen = 0.410
FastText           rata-rata cosine similarity antar dokumen = 0.958


In [60]:
summary = pd.DataFrame({
    'Metode': ['One Hot Encoding', 'Bag of Words', 'TF-IDF', 'Word2Vec (CBOW)', 'FastText (Skip-Gram)'],
    'Jenis fitur': ['sparse, biner', 'sparse, frekuensi', 'sparse, berbobot', 'dense, embedding', 'dense, embedding'],
    'Shape (dokumen x fitur)': [df_ohe.shape, df_bow.shape, df_tfidf.shape, X_w2v.shape, X_ft.shape],
    'Persen nilai nol': [
        f'{(df_ohe.values == 0).mean():.1%}', f'{(df_bow.values == 0).mean():.1%}',
        f'{(df_tfidf.values == 0).mean():.1%}', f'{(X_w2v == 0).mean():.1%}', f'{(X_ft == 0).mean():.1%}'],
})
summary

,Metode,Jenis fitur,Shape (dokumen x fitur),Persen nilai nol
0,One Hot Encoding,"sparse, biner","(8, 65)",85.0%
1,Bag of Words,"sparse, frekuensi","(8, 65)",85.0%
2,TF-IDF,"sparse, berbobot","(8, 65)",85.0%
3,Word2Vec (CBOW),"dense, embedding","(8, 100)",0.0%
4,FastText (Skip-Gram),"dense, embedding","(8, 100)",0.0%


## 6. Menyimpan hasil ke Excel

In [62]:
def with_id(frame):
    out = frame.copy()
    out.insert(0, 'video_id', df['video_id'].values)
    return out

dims = [f'dim_{i}' for i in range(X_w2v.shape[1])]

summary_xlsx = summary.copy()
summary_xlsx['Shape (dokumen x fitur)'] = summary_xlsx['Shape (dokumen x fitur)'].astype(str)

with pd.ExcelWriter('exercise5_feature_engineering.xlsx', engine='openpyxl') as writer:
    summary_xlsx.to_excel(writer, sheet_name='Ringkasan', index=False)
    with_id(df_ohe).to_excel(writer, sheet_name='One Hot Encoding', index=False)
    with_id(df_bow).to_excel(writer, sheet_name='Bag of Words', index=False)
    with_id(df_tfidf).to_excel(writer, sheet_name='TF-IDF', index=False)
    with_id(pd.DataFrame(X_w2v, columns=dims)).to_excel(writer, sheet_name='Word2Vec (vektor dokumen)', index=False)
    with_id(pd.DataFrame(X_ft, columns=dims)).to_excel(writer, sheet_name='FastText (vektor dokumen)', index=False)

print('File berhasil disimpan sebagai exercise5_feature_engineering.xlsx')

try:
    from google.colab import files
    files.download('exercise5_feature_engineering.xlsx')
except ImportError:
    pass

File berhasil disimpan sebagai exercise5_feature_engineering.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>